# Part 2: ETL & Data Preparation (Data Engineer)

## 🎯 Learning Objectives
- Understand ETL (Extract, Transform, Load) process
- Clean datasets **separately** before merging (real-world best practice)
- Implement Bronze → Silver → Gold data layers
- Prepare clean, analysis-ready data

## 📊 Data Architecture (Medallion Pattern)

```
BRONZE (Raw)          SILVER (Cleaned)         GOLD (Business-Ready)
├── reviews.csv   →   ├── clean_reviews.csv   →  final_dataset.csv
├── inventory.csv →   ├── clean_inventory.csv →  (merged + enriched)
└── sales.csv     →   └── clean_sales.csv     →  
```

**Why separate cleaning?**
- Scales better with large datasets
- Each team can own their data source
- Easier to debug issues
- Industry standard (Databricks, Azure, AWS)

In [ ]:
import pandas as pd
import re

print("✅ Libraries loaded")
print("\n📍 Current Phase: BRONZE → SILVER (Cleaning)")

## Step 1: Load Bronze (Raw) Data

In [ ]:
# Load raw data from Bronze layer
reviews_raw = pd.read_csv('../data/raw/reviews.csv')
inventory_raw = pd.read_csv('../data/raw/inventory.csv')
sales_raw = pd.read_csv('../data/raw/sales.csv')

print("📦 Bronze Layer Loaded:")
print(f"  Reviews: {len(reviews_raw):,} rows")
print(f"  Inventory: {len(inventory_raw):,} rows")
print(f"  Sales: {len(sales_raw):,} rows")

---

## 🔍 Step 1.5: Comprehensive Data Quality Assessment

**Before we start cleaning, let's identify ALL the data quality issues.**

This is what you'd do in a real project - audit the data first!

In [ ]:
print("🔍 DATA QUALITY ASSESSMENT")
print("=" * 80)

# REVIEWS DATASET
print("\n📊 REVIEWS.CSV:")
print("-" * 80)
print(f"Total rows: {len(reviews_raw):,}")
print(f"Total columns: {len(reviews_raw.columns)}")

print("\n❌ DATA QUALITY ISSUES FOUND:")

# Issue 1: Missing values
missing_text = reviews_raw['review_text'].isnull().sum()
missing_rating = reviews_raw['rating'].isnull().sum()
print(f"  1. Missing review_text: {missing_text} ({missing_text/len(reviews_raw)*100:.1f}%)")
print(f"  2. Missing rating: {missing_rating} ({missing_rating/len(reviews_raw)*100:.1f}%)")

# Issue 2: Duplicates
duplicates = reviews_raw.duplicated().sum()
print(f"  3. Duplicate rows: {duplicates} ({duplicates/len(reviews_raw)*100:.1f}%)")

# Issue 3: Invalid ratings
invalid_ratings = ((reviews_raw['rating'] < 1) | (reviews_raw['rating'] > 5)).sum()
print(f"  4. Invalid ratings (not 1-5): {invalid_ratings}")
if invalid_ratings > 0:
    print(f"     Examples: {reviews_raw[((reviews_raw['rating'] < 1) | (reviews_raw['rating'] > 5))]['rating'].unique()[:5]}")

# Issue 4: Whitespace
whitespace_issues = reviews_raw['review_text'].astype(str).str.match(r'^\s+|\s+$').sum()
print(f"  5. Reviews with leading/trailing whitespace: ~{whitespace_issues}")

# Issue 5: Date format issues (check for non-standard formats)
date_sample = reviews_raw['date'].astype(str).head(50)
inconsistent_dates = date_sample.str.contains('/').sum()
print(f"  6. Inconsistent date formats detected: {inconsistent_dates} use DD/MM/YYYY instead of YYYY-MM-DD")

# Issue 6: Uppercase text
uppercase_count = reviews_raw['review_text'].astype(str).str.isupper().sum()
print(f"  7. UPPERCASE text: {uppercase_count} reviews")

# INVENTORY DATASET
print("\n📦 INVENTORY.CSV:")
print("-" * 80)
print(f"Total rows: {len(inventory_raw):,}")

print("\n❌ DATA QUALITY ISSUES FOUND:")

missing_stock = inventory_raw['stock_level'].isnull().sum()
print(f"  1. Missing stock_level: {missing_stock} ({missing_stock/len(inventory_raw)*100:.1f}%)")

duplicates_inv = inventory_raw.duplicated().sum()
print(f"  2. Duplicate rows: {duplicates_inv} ({duplicates_inv/len(inventory_raw)*100:.1f}%)")

negative_stock = (inventory_raw['stock_level'] < 0).sum()
print(f"  3. Negative stock values: {negative_stock}")

high_stock = (inventory_raw['stock_level'] > 5000).sum()
print(f"  4. Unrealistically high stock (>5000): {high_stock}")

unknown_products = (inventory_raw['product_name'] == 'Unknown').sum()
print(f"  5. Unknown product names: {unknown_products}")

# SALES DATASET
print("\n💰 SALES.CSV:")
print("-" * 80)
print(f"Total rows: {len(sales_raw):,}")

print("\n❌ DATA QUALITY ISSUES FOUND:")

missing_sales = sales_raw['sales_volume'].isnull().sum()
print(f"  1. Missing sales_volume: {missing_sales} ({missing_sales/len(sales_raw)*100:.1f}%)")

duplicates_sales = sales_raw.duplicated().sum()
print(f"  2. Duplicate rows: {duplicates_sales} ({duplicates_sales/len(sales_raw)*100:.1f}%)")

negative_sales = (sales_raw['sales_volume'] < 0).sum()
print(f"  3. Negative sales: {negative_sales}")

high_sales = (sales_raw['sales_volume'] > 1000).sum()
print(f"  4. Unrealistically high sales (>1000): {high_sales}")

# SUMMARY
print("\n" + "=" * 80)
print("📊 SUMMARY:")
total_issues = (missing_text + missing_rating + duplicates + invalid_ratings + 
                missing_stock + duplicates_inv + negative_stock + 
                missing_sales + duplicates_sales + negative_sales)
print(f"Total data quality issues detected: ~{total_issues}")
print("\n⚠️  This is REALISTIC messy data - just like production!")
print("✅ Now let's clean it step by step...")
print("=" * 80)

### 💭 Observations

**Key findings:**
- Missing values in multiple columns
- Duplicate records across all datasets
- Invalid values (negative stock, invalid ratings)
- Inconsistent formats (dates, text)

**Questions to consider:**
- Can we use data with ~5% missing values?
- Should we fill missing data or drop it?
- How do we handle invalid values?

Now let's clean this data systematically!

In [ ]:
print("🧹 CLEANING REVIEWS DATASET")
print("=" * 70)

# Start with copy of raw data
reviews_clean = reviews_raw.copy()

print(f"Starting rows: {len(reviews_clean):,}")

# 1. Remove rows with missing review text
before = len(reviews_clean)
reviews_clean = reviews_clean.dropna(subset=['review_text'])
print(f"✓ Removed {before - len(reviews_clean)} rows with missing review_text")

# 2. Remove duplicates
before = len(reviews_clean)
reviews_clean = reviews_clean.drop_duplicates()
print(f"✓ Removed {before - len(reviews_clean)} duplicate rows")

# 3. Filter to valid ratings (1-5 only)
before = len(reviews_clean)
reviews_clean = reviews_clean[(reviews_clean['rating'] >= 1) & (reviews_clean['rating'] <= 5)]
print(f"✓ Removed {before - len(reviews_clean)} rows with invalid ratings")

# 4. Clean text: strip whitespace and normalize
reviews_clean['review_text'] = reviews_clean['review_text'].str.strip()
reviews_clean['review_text'] = reviews_clean['review_text'].str.lower()
print(f"✓ Normalized text (lowercase, trimmed whitespace)")

# 5. Parse dates with mixed formats
reviews_clean['date'] = pd.to_datetime(reviews_clean['date'], format='mixed', dayfirst=False, errors='coerce')
before = len(reviews_clean)
reviews_clean = reviews_clean.dropna(subset=['date'])
print(f"✓ Parsed dates, removed {before - len(reviews_clean)} invalid dates")

# 6. Remove future dates
import pandas as pd
before = len(reviews_clean)
reviews_clean = reviews_clean[reviews_clean['date'] <= pd.Timestamp.now()]
print(f"✓ Removed {before - len(reviews_clean)} future dates")

print(f"\nFinal rows: {len(reviews_clean):,}")
print(f"Data reduction: {(1 - len(reviews_clean)/len(reviews_raw))*100:.1f}%")
print("\n✅ Reviews cleaned successfully!")

## Step 3: Clean Inventory Dataset (Bronze → Silver)

In [ ]:
print("🧹 CLEANING INVENTORY DATASET")
print("=" * 70)

inventory_clean = inventory_raw.copy()
print(f"Starting rows: {len(inventory_clean):,}")

# 1. Remove duplicates
before = len(inventory_clean)
inventory_clean = inventory_clean.drop_duplicates()
print(f"✓ Removed {before - len(inventory_clean)} duplicate rows")

# 2. Fill missing stock levels with 0
missing = inventory_clean['stock_level'].isnull().sum()
inventory_clean['stock_level'] = inventory_clean['stock_level'].fillna(0)
print(f"✓ Filled {missing} missing stock_level values with 0")

# 3. Fix negative stock
negative = (inventory_clean['stock_level'] < 0).sum()
inventory_clean.loc[inventory_clean['stock_level'] < 0, 'stock_level'] = 0
print(f"✓ Fixed {negative} negative stock values (set to 0)")

# 4. Cap unrealistically high stock
before_cap = (inventory_clean['stock_level'] > 5000).sum()
inventory_clean.loc[inventory_clean['stock_level'] > 5000, 'stock_level'] = 500
print(f"✓ Capped {before_cap} unrealistically high stock values to 500")

# 5. Remove Unknown products
before = len(inventory_clean)
inventory_clean = inventory_clean[inventory_clean['product_name'] != 'Unknown']
print(f"✓ Removed {before - len(inventory_clean)} rows with Unknown products")

# 6. Parse dates
inventory_clean['date'] = pd.to_datetime(inventory_clean['date'], format='mixed', errors='coerce')
inventory_clean = inventory_clean.dropna(subset=['date'])

print(f"\nFinal rows: {len(inventory_clean):,}")
print("✅ Inventory cleaned successfully!")

## Step 4: Clean Sales Dataset (Bronze → Silver)

In [ ]:
print("🧹 CLEANING SALES DATASET")
print("=" * 70)

sales_clean = sales_raw.copy()
print(f"Starting rows: {len(sales_clean):,}")

# 1. Remove duplicates
before = len(sales_clean)
sales_clean = sales_clean.drop_duplicates()
print(f"✓ Removed {before - len(sales_clean)} duplicate rows")

# 2. Fill missing sales_volume with 0
missing = sales_clean['sales_volume'].isnull().sum()
sales_clean['sales_volume'] = sales_clean['sales_volume'].fillna(0)
print(f"✓ Filled {missing} missing sales_volume with 0")

# 3. Fix negative sales
negative = (sales_clean['sales_volume'] < 0).sum()
sales_clean.loc[sales_clean['sales_volume'] < 0, 'units_sold'] = 0
print(f"✓ Fixed {negative} negative sales (set to 0)")

# 4. Cap unrealistically high sales
before_cap = (sales_clean['sales_volume'] > 1000).sum()
sales_clean.loc[sales_clean['sales_volume'] > 1000, 'units_sold'] = 200
print(f"✓ Capped {before_cap} unrealistically high sales to 200")

# 5. Parse dates
sales_clean['date'] = pd.to_datetime(sales_clean['date'], format='mixed', errors='coerce')
sales_clean = sales_clean.dropna(subset=['date'])

print(f"\nFinal rows: {len(sales_clean):,}")
print("✅ Sales cleaned successfully!")

---

## Step 5: Save to SILVER Layer

### 📊 Data Storage Layers Explained:

**🥉 BRONZE Layer** (Raw Data)
- Original data exactly as received
- Immutable (never modified)
- Contains all quality issues
- Purpose: Audit trail, ability to re-process

**🥈 SILVER Layer** (Cleaned Data)
- Validated and cleaned
- Duplicates removed
- Data types corrected
- Still separate tables (not merged yet)
- Purpose: Reusable clean datasets

**🥇 GOLD Layer** (Business-Ready)
- Merged and enriched
- Aggregated for specific use cases
- Ready for ML models and analytics
- Purpose: Direct consumption by data scientists

This is the **Medallion Architecture**

Let's save our clean data to SILVER!

In [ ]:
import os
# Create silver directory if it doesn't exist
os.makedirs('../data/silver', exist_ok=True)
print("💾 SAVING TO SILVER LAYER")
print("=" * 70)
# Save cleaned datasets
reviews_clean.to_csv('../data/silver/reviews_clean.csv', index=False)
print(f"✓ Saved reviews_clean.csv ({len(reviews_clean):,} rows)")

inventory_clean.to_csv('../data/silver/inventory_clean.csv', index=False)
print(f"✓ Saved inventory_clean.csv ({len(inventory_clean):,} rows)")

sales_clean.to_csv('../data/silver/sales_clean.csv', index=False)
print(f"✓ Saved sales_clean.csv ({len(sales_clean):,} rows)")

print("\n" + "=" * 70)
print("✅ SILVER LAYER COMPLETE!")
print("=" * 70)
print("\nData is now:")
print("  ✓ Validated and cleaned")
print("  ✓ Duplicates removed")
print("  ✓ Invalid values fixed")
print("  ✓ Formats standardized")
print("  ✓ Ready for merging into GOLD layer")

---

## Step 6: Create GOLD Dataset (Merge & Enrich)

Now we'll merge all three clean datasets into one **business-ready** dataset.

### What we're creating:
- One row per review
- Enriched with stock levels (from inventory)
- Enriched with sales data (from sales)
- Ready for ML model training

In [ ]:
print("🥇 CREATING GOLD DATASET")
print("=" * 70)

# Start with clean reviews
gold_df = reviews_clean.copy()
print(f"Starting with reviews: {len(gold_df):,} rows")

# Aggregate inventory by product (average stock level)
inventory_agg = inventory_clean.groupby('product_id').agg({
    'stock_level': 'mean'
}).reset_index()
inventory_agg.columns = ['product_id', 'avg_stock_level']
print(f"\n✓ Aggregated inventory: {len(inventory_agg)} products")

# Aggregate sales by product (average daily sales)
sales_agg = sales_clean.groupby('product_id').agg({
    'units_sold': 'mean'
}).reset_index()
sales_agg.columns = ['product_id', 'avg_daily_sales']
print(f"✓ Aggregated sales: {len(sales_agg)} products")

# Merge inventory data
gold_df = gold_df.merge(inventory_agg, on='product_id', how='left')
print(f"\n✓ Merged inventory data")

# Merge sales data
gold_df = gold_df.merge(sales_agg, on='product_id', how='left')
print(f"✓ Merged sales data")

# Create additional features
gold_df['stock_to_sales_ratio'] = gold_df['avg_stock_level'] / (gold_df['avg_daily_sales'] + 1)
gold_df['days_of_inventory'] = gold_df['stock_to_sales_ratio']
print(f"✓ Created derived features")


# Create sentiment labels from ratings
def rating_to_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:  # 4 or 5
        return 'positive'

gold_df['sentiment'] = gold_df['rating'].apply(rating_to_sentiment)
print(f"✓ Created sentiment labels from ratings")

# Rename review_text to review_text_clean for clarity
gold_df['review_text_clean'] = gold_df['review_text']
print(f"✓ Renamed review_text to review_text_clean")

# Show final dataset info
print("\n" + "=" * 70)
print("📊 GOLD DATASET INFO:")
print("=" * 70)
print(f"Total rows: {len(gold_df):,}")
print(f"Total columns: {len(gold_df.columns)}")
print(f"\nColumns: {list(gold_df.columns)}")
print(f"\nSample data:")
print(gold_df.head(3))

print("\n✅ GOLD dataset created successfully!")

## Step 7: Save GOLD Dataset

This is the final, **ML-ready** dataset that will be used in Notebook 3 for model training.

In [ ]:
# Create gold directory
os.makedirs('../data/gold', exist_ok=True)

# Save GOLD dataset
gold_df.to_csv('../data/gold/final_dataset.csv', index=False)

print("💾 SAVED TO GOLD LAYER")
print("=" * 70)
print(f"✓ Saved final_dataset.csv ({len(gold_df):,} rows)")
print(f"✓ Location: data/gold/final_dataset.csv")
print("\n" + "=" * 70)
print("✅ ETL PIPELINE COMPLETE!")
print("=" * 70)
print("\nData Journey:")
print("  🥉 BRONZE → Raw data with quality issues")
print("  🥈 SILVER → Cleaned and validated data")
print("  🥇 GOLD   → Merged, enriched, ML-ready data")
print("\nNext: Notebook 3 - ML Model Training 🚀")

---

## 💡 Key Takeaways: ETL & Data Preparation

### What We Accomplished:

1. **Data Quality Assessment** 🔍
   - Identified ~150+ data quality issues
   - Understood the scope before cleaning


2. **Systematic Cleaning** 🧹
   - Removed missing values (where appropriate)
   - Eliminated duplicates
   - Fixed invalid values
   - Standardized formats


3. **Medallion Architecture** 🏅
   - BRONZE: Immutable raw data (audit trail)
   - SILVER: Clean, validated data (reusable)
   - GOLD: Business-ready data (ML/analytics)


4. **Data Enrichment** ✨
   - Merged three datasets
   - Added calculated features
   - Created analysis-ready dataset


**Why this matters:**
- ✅ Data scientists get clean data immediately
- ✅ Can always re-process from BRONZE if needed
- ✅ SILVER data reusable across projects
- ✅ Clear ownership and lineage

**Next Step**: Notebook 3 - Train ML model on the GOLD dataset! 🚀